In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SILVER = f"{CATALOG}.rc_silver.survey_clean"
GOLD = f"{CATALOG}.rc_gold"
UNKNOWN_KEY, UNKNOWN_LABEL = -1, "Non renseigné"

sv = spark.table(SILVER)
print("Années présentes dans Silver :", sorted(r[0] for r in sv.select("_survey_year").distinct().collect()))

AGE = ["18-24 years old", "25-34 years old", "35-44 years old", "45-54 years old",
       "55-64 years old", "65 years or older", "Prefer not to say"]
AI_USAGE = ["Yes, I use AI tools daily", "Yes, I use AI tools weekly",
            "Yes, I use AI tools monthly or infrequently", "No, but I plan to soon",
            "No, and I don't plan to"]
AI_TRUST = ["Highly trust", "Somewhat trust", "Neither trust nor distrust",
            "Somewhat distrust", "Highly distrust"]
AI_SENT = ["Very favorable", "Favorable", "Indifferent", "Unsure", "Unfavorable", "Very unfavorable"]
BANDS = ["1-5 years", "6-10 years", "11-15 years", "16-20 years",
         "21-30 years", "31-40 years", "41-50 years", "More than 50 years"]
BRANCH_SHORT = {
    "I am a developer by profession": "Professional developer",
    "I used to be a developer by profession, but no longer am": "Former professional developer",
    "I am not primarily a developer, but I write code sometimes as part of my work/studies": "Occasional coder (work/studies)",
    "I am learning to code": "Learning to code",
    "I code primarily as a hobby": "Hobbyist",
    "I work with developers or my work supports developers but am not a developer by profession": "Supports developers",
}

def make_dim(name, src_col, order=None, short=None):
    key, label = f"{name}_key", f"{name}_label"
    present = ({r[0] for r in sv.where(F.col(src_col).isNotNull()).select(src_col).distinct().collect()}
               if src_col else set())
    if order is None:
        order = sorted(present)
    elif src_col:
        unexpected = present - set(order)
        assert not unexpected, f"CRITIQUE : libellés inattendus pour {name} : {sorted(unexpected)[:5]}"
    rows = [(i, l, (short or {}).get(l, l), i) for i, l in enumerate(order, 1)]
    rows.append((UNKNOWN_KEY, UNKNOWN_LABEL, UNKNOWN_LABEL, 999999))
    df = spark.createDataFrame(rows, f"{key} int, {label} string, {name}_short string, sort_order int")
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(f"{GOLD}.dim_{name}"))
    print(f"dim_{name} : {df.count()} lignes")

make_dim("main_branch", "MainBranch", order=list(BRANCH_SHORT), short=BRANCH_SHORT)
make_dim("age", "Age", order=AGE)
make_dim("country", "Country")
make_dim("dev_type", "DevType")
make_dim("work_exp_band", None, order=BANDS)
make_dim("years_code_band", None, order=BANDS)
make_dim("ai_usage", "AISelect", order=AI_USAGE)
make_dim("ai_trust", "AIAcc", order=AI_TRUST)
make_dim("ai_sentiment", "AISent", order=AI_SENT)

Années présentes dans Silver : [2025]
dim_main_branch : 7 lignes
dim_age : 8 lignes
dim_country : 178 lignes
dim_dev_type : 33 lignes
dim_work_exp_band : 9 lignes
dim_years_code_band : 9 lignes
dim_ai_usage : 6 lignes
dim_ai_trust : 6 lignes
dim_ai_sentiment : 7 lignes


In [0]:
def band(col):
    c = F.col(col)
    return (F.when(c.isNull(), F.lit(None).cast("string"))
             .when(c <= 5, "1-5 years").when(c <= 10, "6-10 years").when(c <= 15, "11-15 years")
             .when(c <= 20, "16-20 years").when(c <= 30, "21-30 years").when(c <= 40, "31-40 years")
             .when(c <= 50, "41-50 years").otherwise("More than 50 years"))

def flag(col, labels):
    return F.when(F.col(col).isNull(), F.lit(None).cast("boolean")).otherwise(F.col(col).isin(labels))

# (dimension, colonne source dans Silver, colonne source dans la table de faits)
DIMS = [("main_branch", "MainBranch", "MainBranch"), ("age", "Age", "Age"),
        ("country", "Country", "Country"), ("dev_type", "DevType", "DevType"),
        ("work_exp_band", "WorkExp", "work_exp_band"), ("years_code_band", "YearsCode", "years_code_band"),
        ("ai_usage", "AISelect", "AISelect"), ("ai_trust", "AIAcc", "AIAcc"),
        ("ai_sentiment", "AISent", "AISent")]

f0 = (sv.select(F.col("_survey_year").alias("survey_year"), "ResponseId", "MainBranch", "Age", "Country", "DevType",
                F.col("WorkExp").alias("work_exp"), F.col("YearsCode").alias("years_code"),
                F.col("JobSat").alias("job_sat"), F.col("ConvertedCompYearly").alias("comp_yearly_usd"),
                "AISelect", "AIAcc", "AISent",
                "is_professional", "is_low_effort", "is_content_duplicate", "is_salary_extreme")
        .withColumn("work_exp_band", band("work_exp"))
        .withColumn("years_code_band", band("years_code"))
        .withColumn("ai_user", flag("AISelect", AI_USAGE[:3]))
        .withColumn("ai_user_or_planning", flag("AISelect", AI_USAGE[:4]))
        .withColumn("ai_trusts", flag("AIAcc", AI_TRUST[:2]))
        .withColumn("ai_distrusts", flag("AIAcc", AI_TRUST[3:]))
        .withColumn("ai_sentiment_positive", flag("AISent", AI_SENT[:2]))
        .withColumn("job_sat_group",
                    F.when(F.col("job_sat").isNull(), F.lit(None).cast("string"))
                     .when(F.col("job_sat") <= 6, "Not happy (0-6)")
                     .when(F.col("job_sat") <= 8, "Complacent (7-8)")
                     .otherwise("Happy (9-10)")))

def add_key(df, src_col, dim_name):
    key, label = f"{dim_name}_key", f"{dim_name}_label"
    d = spark.table(f"{GOLD}.dim_{dim_name}").select(F.col(key), F.col(label).alias("_lbl"))
    return (df.join(F.broadcast(d), df[src_col] == d["_lbl"], "left")
              .withColumn(key, F.coalesce(F.col(key), F.lit(UNKNOWN_KEY))).drop("_lbl"))

fact = f0
for _, _, fact_src in DIMS:
    dim = [d for d, _, s in DIMS if s == fact_src][0]
    fact = add_key(fact, fact_src, dim)

fact = fact.drop("MainBranch", "Age", "Country", "DevType", "work_exp_band", "years_code_band",
                 "AISelect", "AIAcc", "AISent")
(fact.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
     .saveAsTable(f"{GOLD}.fact_respondent"))
print("fact_respondent :", spark.table(f"{GOLD}.fact_respondent").count(), "lignes")

fact_respondent : 49191 lignes


In [0]:
fr = spark.table(f"{GOLD}.fact_respondent")
n_f, n_s = fr.count(), sv.count()
print(f"Lignes : fact {n_f:,} | Silver {n_s:,}")
assert n_f == n_s, "CRITIQUE : nombre de lignes"
assert fr.select("ResponseId").distinct().count() == n_f, "CRITIQUE : ResponseId dupliqué"

for dim, silver_col, _ in DIMS:
    key = f"{dim}_key"
    orphans = fr.join(spark.table(f"{GOLD}.dim_{dim}").select(key), key, "left_anti").count()
    unknown = fr.where(F.col(key) == UNKNOWN_KEY).count()
    expected = sv.where(F.col(silver_col).isNull()).count()
    print(f"dim_{dim:<16} orphelins={orphans} | 'Non renseigné'={unknown:,} (NULL Silver : {expected:,})")
    assert orphans == 0 and unknown == expected, f"CRITIQUE : {dim}"

# Indicateurs recoupés avec les comptages de l'EDA (valables pour 2025)
agg = fr.agg(*[F.sum(F.col(c).cast("int")).alias(c) for c in
               ["ai_user", "ai_user_or_planning", "ai_trusts", "ai_distrusts", "ai_sentiment_positive"]]).collect()[0].asDict()
expected_flags = {"ai_user": 26469, "ai_user_or_planning": 28266, "ai_trusts": 10917,
                  "ai_distrusts": 15218, "ai_sentiment_positive": 19988}
for c, v in expected_flags.items():
    print(f"{c:<24} {agg[c]:,} (attendu {v:,})")
    assert agg[c] == v, f"CRITIQUE : {c}"

js = {r["job_sat_group"]: r["count"] for r in fr.groupBy("job_sat_group").count().collect()}
print({k: v for k, v in js.items() if k})
assert (js.get("Not happy (0-6)"), js.get("Complacent (7-8)"), js.get("Happy (9-10)")) == (7586, 12562, 6522), "CRITIQUE : job_sat_group"
print("Validation Gold (modèle) : OK")

Lignes : fact 49,191 | Silver 49,191
dim_main_branch      orphelins=0 | 'Non renseigné'=0 (NULL Silver : 0)
dim_age              orphelins=0 | 'Non renseigné'=0 (NULL Silver : 0)
dim_country          orphelins=0 | 'Non renseigné'=13,754 (NULL Silver : 13,754)
dim_dev_type         orphelins=0 | 'Non renseigné'=5,511 (NULL Silver : 5,511)
dim_work_exp_band    orphelins=0 | 'Non renseigné'=6,298 (NULL Silver : 6,298)
dim_years_code_band  orphelins=0 | 'Non renseigné'=6,149 (NULL Silver : 6,149)
dim_ai_usage         orphelins=0 | 'Non renseigné'=15,471 (NULL Silver : 15,471)
dim_ai_trust         orphelins=0 | 'Non renseigné'=15,894 (NULL Silver : 15,894)
dim_ai_sentiment     orphelins=0 | 'Non renseigné'=15,724 (NULL Silver : 15,724)
ai_user                  26,469 (attendu 26,469)
ai_user_or_planning      28,266 (attendu 28,266)
ai_trusts                10,917 (attendu 10,917)
ai_distrusts             15,218 (attendu 15,218)
ai_sentiment_positive    19,988 (attendu 19,988)
{'Happy (9-10)'